# CHALLENGE 3: Streaming a Real Reach Trajectory to a Resource-Constrained Device

## Course architecture

Input design ➡️ On-device input processor ➡️ On-device output processor ➡️ Resource usage logger

**Optional Dataset Challenge — Colab 3 flavor (streaming / resource-constrained edge simulation).** This notebook mirrors Lab Stage 3's structure exactly, using a different, real data source in place of the simulated temperature sensor: the **[O'Doherty/Sabes Nonhuman Primate Reaching dataset](https://zenodo.org/records/583331)** (Zenodo, CC-BY 4.0) — continuous 2D hand-position recordings from a macaque monkey performing a reaching task, recorded alongside multichannel motor-cortex electrophysiology. We'll only use the 2D position channel here (`cursor_pos`, sampled at 250 Hz) — same engineering challenge as Lab Stage 3 (bounded-memory streaming, real-time output, resource logging), but the stream now carries a real behavioral signal instead of a simulated one.

> ℹ️ The session file used here (`indy_20160927_06.mat`, about 127 MB) and the `cursor_pos` and `t` fields are listed on the [Zenodo record page](https://zenodo.org/records/583331) (checked 24 Nov 2026). The download takes a minute or two on a good connection.

## Instructions for the student

As in Lab Stage 3, you'll adapt a `MicroBit` simulator class — 128 KB of RAM, no `numpy`/`scipy` on the "device" itself — one method at a time, passing each "AI Prompt Challenge" to an AI coding assistant. This is an **optional** extension for anyone who wants practice with a real primate electrophysiology dataset and a different repository (Zenodo) than Lab Stage 1 and 2 used.

> ✅ **Ask → Check → Build → Document.** Before running anything an AI tool gives you, check it with three questions: Does it do what I asked? Does it do anything I didn't ask for, such as downloading or deleting files or sending data anywhere? Can I explain each step in plain words? If any answer is no, ask the AI to explain or fix it first. Never paste identifiable participant data, passwords or access keys into a public AI tool.

***
## 🔧 Starting Point: The `MicroBit` Simulator Class

Same class as Lab Stage 3, given as a skeleton — three methods left as `pass` stubs for you to fill in across Modules 2–4: `read_from_position_sensor`, `process_position_data`, and `to_file`.

**Given (do not modify `__init__`, `ram_used`, `_is_ram_available`, or `output_to_screen`):**

In [ ]:
import numpy as np
import time
from datetime import datetime
from IPython.display import clear_output

class MicroBit:

    def __init__(self):
        super().__init__()

        self.MAX_RAM = 128 * 1024      # (bytes)
        self.MAX_STORAGE = 512 * 1024  # (bytes)

        self.buffer = []
        self.storage = []
        self.BYTES_PER_SAMPLE = 16     # one (x, y) pair of 8-byte numbers

    def ram_used(self):
        return len(self.buffer) * self.BYTES_PER_SAMPLE

    def _is_ram_available(self, extra_bytes=16):
        # Would storing `extra_bytes` more still fit? Check BEFORE storing.
        return self.ram_used() + extra_bytes <= self.MAX_RAM

    def output_to_screen(self, positions):
        clear_output(wait=True)
        grid = np.full((5, 5), '.', dtype=str)

        for r, c in positions:
            grid[r, c] = '*'

        for row in grid:
            print(' '.join(row))

    def read_from_position_sensor(self, input_stream):
        pass  # TODO: Module 2

    def process_position_data(self):
        pass  # TODO: Module 3

    def to_file(self, file):
        pass  # TODO: Module 4

***
## 📡 Module 1: Input Design

Instead of generating a simulated signal, we load a real recording session and prepare it as a stream of (x, y) hand-position tuples — as if they were arriving live from the recording rig, one sample at a time.

**Setup (given):**

In [ ]:
!wget -q -O indy_session.mat "https://zenodo.org/records/583331/files/indy_20160927_06.mat"

from scipy.io import loadmat

try:
    mat = loadmat("indy_session.mat", variable_names=["cursor_pos", "t"])
    cursor_pos = mat["cursor_pos"]   # (n_samples, 2) — x, y cursor position in mm
    t = mat["t"].flatten()           # (n_samples,) — timestamps in seconds, ~250 Hz
except NotImplementedError:          # MATLAB v7.3 files are HDF5: read them with h5py instead
    import h5py
    with h5py.File("indy_session.mat", "r") as f:
        cursor_pos = f["cursor_pos"][()]
        t = f["t"][()].flatten()
    if cursor_pos.shape[0] == 2 and cursor_pos.shape[1] != 2:
        cursor_pos = cursor_pos.T    # HDF5 stores MATLAB arrays transposed

print(f"Loaded {cursor_pos.shape[0]} position samples spanning {t[-1] - t[0]:.1f} seconds")

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "I have a numpy array cursor_pos of shape (n_samples, 2) holding (x, y) hand positions. Write code that builds a list called position_stream containing each row as an (x, y) tuple, in order, so it can be read one sample at a time — the same way a live sensor would deliver it."


In [ ]:
# TODO: Your code here

In [ ]:
print(len(position_stream))
print(position_stream[0])

***
## 📥 Module 2: On-Device Input Processor

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "Write the method read_from_position_sensor assuming input_stream is a list of (x, y) tuples. The method should read the list element-by-element and, for each value, call self._is_ram_available() *before* storing it; only if there is room, append the value to self.buffer. It should print '[ERROR]: Out of RAM!' if no more RAM is available and return 1. If it managed to successfully read the whole stream, it should print f'[SUCCESS] Used RAM: {self.ram_used()} bytes.' and return 0."

Paste the solution into the `read_from_position_sensor` method in the class above, then re-run that cell before continuing.


In [ ]:
microbit = MicroBit()
microbit.read_from_position_sensor(position_stream)

***
## 🖥️ Module 3: On-Device Output Processor

The 5×5 LED grid can't show a continuous (x, y) coordinate directly — we need to bin each position into one of 25 grid cells.

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "Write the method process_position_data. It should read self.buffer value-by-value, where each value is an (x, y) tuple. Using the minimum and maximum x and y across the whole buffer, bin each position into a 5x5 grid (5 evenly spaced bins per axis) and call self.output_to_screen with the resulting list of (row, col) positions."

Paste the solution into the `process_position_data` method in the class above, then re-run that cell before continuing.


In [ ]:
microbit.process_position_data()

***
## 📝 Module 4: Resource Usage Logger

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "Write the method to_file which takes a file name and appends the current time (datetime.now().time()) and the current RAM used. If the file does not exist, the method should automatically create it."

Paste the solution into the `to_file` method in the class above, then re-run that cell before continuing.


In [ ]:
microbit.to_file("out.txt")

***
## 🎮 Embedded playground

**Task 1.** The full session has far more samples than 128 KB of RAM can hold at once. Feed the *entire* `position_stream` (not a slice of it) into a fresh `MicroBit()` instance and observe the `[ERROR]: Out of RAM!` behavior firsthand. Roughly how many samples get through before it triggers?

**Task 2.** Modify `read_from_position_sensor` to print the running RAM usage every 1,000 samples while it reads, instead of only printing once at the end.

**Task 3.** The 5×5 grid is coarse. Modify `process_position_data` (and `output_to_screen`, if needed) to use a 10×10 grid instead, and compare how much more of the reach trajectory's structure you can see.

**Task 4.** The class already has `self.storage` and `self.MAX_STORAGE` defined but unused. Prompt your AI assistant to add a method that persists each position read into `self.storage` (respecting `MAX_STORAGE`, the same way `read_from_position_sensor` respects `MAX_RAM`), so the device keeps a longer-term record than the working buffer alone.

In [ ]:
microbit = MicroBit()
microbit.read_from_position_sensor(position_stream)
microbit.process_position_data()
microbit.to_file("out.txt")